# Week 6: The Win-Back Equation — Should You Chase Lost Customers?
## Propensity-to-Return Modeling & ROI Framework

**Masterclass:** [Week 6 Deep-Dive](https://balaviswanath-analytics.github.io/analytics_masterclass/week6_winback_equation.html)

**Dataset:** CDNOW transactions (bundled with `lifetimes`) + synthetic win-back outcomes

**What You'll Build:**
1. Identify churned customers from transaction data
2. Build a propensity-to-return model (logistic regression)
3. Win-back ROI framework: cost vs. expected recovered CLV
4. Prioritization matrix: who's worth chasing?

In [ ]:
!pip install lifetimes lifelines pandas matplotlib seaborn scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifetimes.datasets import load_transaction_data
from lifetimes.utils import summary_data_from_transaction_data
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print("Libraries loaded.")

---
## Part 1: Identify Churned Customers

In [ ]:
txns = load_transaction_data()
cutoff = txns['date'].max()

rfm = summary_data_from_transaction_data(txns, 'id', 'date',
    monetary_value_col='spent', observation_period_end=cutoff)

# Define churn: no purchase in last 90 days
rfm['days_since_last'] = (rfm['T'] - rfm['recency']) * 7  # convert weeks to days approx
rfm['churned'] = (rfm['days_since_last'] > 90).astype(int)

churned = rfm[rfm['churned'] == 1].copy()
print(f"Total customers: {len(rfm):,}")
print(f"Churned (>90 days inactive): {len(churned):,} ({len(churned)/len(rfm):.1%})")

---
## Part 2: Simulate Win-Back Campaign Outcomes

In a real scenario, you'd have historical win-back campaign data. Here we simulate realistic outcomes: customers with higher historical frequency and monetary value are more likely to return.

In [ ]:
np.random.seed(42)

# Simulate: P(return) depends on frequency, recency gap, and monetary value
churned['recency_gap'] = churned['T'] - churned['recency']
logit = (-2.0
         + 0.3 * np.log1p(churned['frequency'])
         + 0.2 * np.log1p(churned['monetary_value'])
         - 0.15 * churned['recency_gap'])
churned['p_return_true'] = 1 / (1 + np.exp(-logit))
churned['returned'] = (np.random.rand(len(churned)) < churned['p_return_true']).astype(int)

print(f"Win-back success rate: {churned['returned'].mean():.1%}")
print(f"Returned: {churned['returned'].sum()}, Did not return: {(1-churned['returned']).sum():.0f}")

---
## Part 3: Propensity-to-Return Model

In [ ]:
features = ['frequency', 'recency', 'T', 'monetary_value', 'recency_gap']
X = churned[features].copy()
y = churned['returned']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lr = LogisticRegression(random_state=42, max_iter=1000)
scores = cross_val_score(lr, X_scaled, y, cv=5, scoring='roc_auc')
print(f"Cross-validated AUC: {scores.mean():.3f} (+/- {scores.std():.3f})")

lr.fit(X_scaled, y)
churned['p_return'] = lr.predict_proba(X_scaled)[:, 1]

# Feature importance
print("\n=== Feature Coefficients ===")
for feat, coef in sorted(zip(features, lr.coef_[0]), key=lambda x: -abs(x[1])):
    direction = 'increases' if coef > 0 else 'decreases'
    print(f"  {feat}: {coef:+.3f} ({direction} return probability)")

---
## Part 4: Win-Back ROI Framework

In [ ]:
# Assumptions
WINBACK_COST = 25       # cost per win-back attempt (email + offer)
AVG_2ND_LIFE_CLV = 150  # average CLV of a successfully reactivated customer
MARGIN = 0.40

churned['expected_value'] = churned['p_return'] * AVG_2ND_LIFE_CLV * MARGIN
churned['expected_profit'] = churned['expected_value'] - WINBACK_COST
churned['worth_pursuing'] = churned['expected_profit'] > 0

print("=== Win-Back ROI Summary ===")
print(f"Total churned: {len(churned):,}")
print(f"Worth pursuing (E[profit] > 0): {churned['worth_pursuing'].sum():,} ({churned['worth_pursuing'].mean():.1%})")
print(f"Not worth it: {(~churned['worth_pursuing']).sum():,}")
print(f"\nExpected total profit (pursue all worth it): ${churned.loc[churned['worth_pursuing'], 'expected_profit'].sum():,.0f}")
print(f"Expected waste (pursue all NOT worth it): ${-churned.loc[~churned['worth_pursuing'], 'expected_profit'].sum():,.0f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(churned['p_return'], bins=30, color='#264653', edgecolor='white', alpha=0.85)
axes[0].axvline(x=WINBACK_COST / (AVG_2ND_LIFE_CLV * MARGIN), color='#c45d3e',
                linestyle='--', linewidth=2, label='Break-even P(return)')
axes[0].set_title('Distribution of P(Return)')
axes[0].set_xlabel('Predicted P(Return)')
axes[0].legend()

axes[1].scatter(churned['p_return'], churned['expected_profit'],
                c=churned['worth_pursuing'].map({True: '#2d6a4f', False: '#c45d3e'}),
                alpha=0.4, s=15)
axes[1].axhline(y=0, color='gray', linestyle='--')
axes[1].set_title('Expected Profit per Customer')
axes[1].set_xlabel('P(Return)')
axes[1].set_ylabel('Expected Profit ($)')
plt.tight_layout()
plt.show()

---
## Exercises

### Exercise 1: Vary the Offer
Recompute ROI with WINBACK_COST = $50 (richer offer). Does the "worth pursuing" pool shrink or grow? At what cost does it become unprofitable to pursue anyone?

### Exercise 2: Segment the Win-Back List
Split churned customers into High/Medium/Low CLV tiers. What's the optimal win-back strategy for each tier?

### Exercise 3: Time Decay
Add "months since churn" as a feature. Does the propensity to return decay with time? At what point does it become near-zero?

---
## Key Takeaways
1. **Not all churned customers are worth chasing** — the ROI math is unforgiving
2. **Propensity models** prevent wasting budget on customers who won't return
3. **Break-even P(return)** = cost / (CLV x margin) — the minimum threshold
4. **Time decay** makes early intervention critical